# goldlabel walkthrough

Eight short posts with hand-built 2-D vectors live in `corpus.jsonl`. Every cell below is one step of
the analysis; run them in order, in Jupyter, marimo, or as a plain script (`# %%` marks the cells).

What needs you: a classifier model and its key (`OPENAI_API_KEY` in the environment; nothing
here reads or writes a key file), and **a person** to label the gold rows in the labelling page.
Everything before the classifier runs offline. When the script has run, `uv run goldlabel label`
(from this folder) opens the same page on every dataset, to browse the rows, read the evaluations
and revise a label.

In [1]:
from pydantic_ai import Embedder
from pydantic_ai.embeddings import EmbeddingResult, TestEmbeddingModel
from pydantic_ai.usage import RequestUsage

import goldlabel as gl


class HandEmbedder(Embedder):
    """Stand-in for a real embedding model: the sample vectors are hand-built, not embeddings.

    With your own corpus, drop this and pass `embedder="openai:text-embedding-3-small"` (or the
    model your corpus was embedded with) to `gl.get_candidates`.
    """

    def __init__(self):
        super().__init__(TestEmbeddingModel("hand-built-2d"))

    async def embed_query(self, query, *, settings=None):
        return EmbeddingResult(
            [[1.0, 0.0]],
            inputs=[query],
            input_type="query",
            model_name="hand-built-2d",
            provider_name="test",
            usage=RequestUsage(input_tokens=1),
        )

## 1. Search

Seed phrases are things a matching post could say. `get_candidates` embeds them, finds the closest items
and keeps each item's best similarity. The result is saved to `.goldlabel/candidates.jsonl`.

In [2]:
corpus = gl.Table(
    "corpus.jsonl",
    id_column="id",
    vector_column="embedding",
    embedding_model="hand-built-2d",
)
seeds = ["I just lost my job"]
candidates = gl.get_candidates(corpus, seeds=seeds, embedder=HandEmbedder())
candidates

id,text,similarity,best seed
layoff-1,I got laid off on Monday after nine years at the company.,0.950,I just lost my job
layoff-2,"My whole team was cut in the reorg, and I am job hunting now.",0.900,I just lost my job
layoff-3,Rumours of redundancies are going around the office.,0.800,I just lost my job
burnout-1,I am exhausted and dread opening my laptop every morning.,0.700,I just lost my job
burnout-2,Working weekends for a year has finally caught up with me.,0.660,I just lost my job


In [3]:
candidates.to_polars()  # .to_pandas() works the same way

id,text,max_similarity,best_seed
str,str,f64,str
"""layoff-1""","""I got laid off on Monday after…",0.95,"""I just lost my job"""
"""layoff-2""","""My whole team was cut in the r…",0.9,"""I just lost my job"""
"""layoff-3""","""Rumours of redundancies are go…",0.8,"""I just lost my job"""
"""burnout-1""","""I am exhausted and dread openi…",0.7,"""I just lost my job"""
"""burnout-2""","""Working weekends for a year ha…",0.66,"""I just lost my job"""


## 2. Labels and a first prompt

`single` means one label per post; `off_topic` is built in. The classifier is immutable: a new
prompt is a new classifier (`with_prompt`), so you can keep both and compare them.

In [4]:
taxonomy = gl.Taxonomy(
    mode="single",
    labels={
        "layoff": "The author lost their job, or expects to.",
        "burnout": "The author is worn out by their work.",
    },
)
clf = gl.Classifier(
    "openai:gpt-6-luna",
    taxonomy,
    prompt="Read the post and choose the one label that fits it best.",
)
clf

Classifier(model='openai:gpt-6-luna', taxonomy=Taxonomy(mode='single', labels=[Label(name='layoff', description='The author lost their job, or expects to.'), Label(name='burnout', description='The author is worn out by their work.')]), prompt='Read the post and choose the one label that fits it best.', cache=None)

## 3. Gold: people label

`draw_sample` picks pending rows, with no labels, and is safe to rerun: it draws only what is
missing. `gl.label` opens the labelling page in your browser and
returns when you press **Done**. You decide every label; the page never shows the classifier's
guess. (An agent working with you must not fill these in, see the README.) Label the three posts
by what they say: the first is a layoff, the other two are burnout.

In [5]:
gold = gl.Gold(candidates)
dev = gold.draw_sample("dev", n=3)

In [6]:
gl.label(dev, taxonomy)

All 3 rows in dev are labelled; nothing to label. Pass force_open_ui=True to review or relabel.


id,text,labels,sample,labelled by
layoff-1,I got laid off on Monday after nine years at the company.,layoff,dev,goldlabel-ui
burnout-2,Working weekends for a year has finally caught up with me.,burnout,dev,goldlabel-ui
burnout-1,I am exhausted and dread opening my laptop every morning.,burnout,dev,goldlabel-ui


## 4. Evaluate on the dev split

One model call per labelled row; running the same cell again makes none (they are cached).
Every evaluation is also appended to `.goldlabel/evals.jsonl`, with the rows it got wrong. Run
`uv run goldlabel label` and open the "dev" sample to read them there: change a label you got
wrong, and the evaluation is marked STALE until you run it again.

In [7]:
ev = clf.evaluate(dev, target=0.9)
ev

3/3  0 live, 3 cached, 0 errors, cost $0.0000


Evaluation: 3 rows, 0 pending rows skipped, 0 disagreements
accuracy 1.0000 PASS (target 0.9000)
macro_f1 1.0000, micro_f1 1.0000
label      precision  recall  f1    gold  predicted
layoff          1.00    1.00  1.00     1          1
burnout         1.00    1.00  1.00     2          2
off_topic       0.00    0.00  0.00     0          0
sources: goldlabel-ui 3, imported 0, unstamped 0

In [8]:
ev.disagreements_to_polars()  # where the classifier and the people differ, with its reasoning

shape: (0, 0)
┌┐
╞╡
└┘

## 5. Tune the prompt on dev, never on test

Change one thing, build a new classifier, evaluate again. Only the changed prompt costs calls.

In [9]:
clf2 = clf.with_prompt(
    "Read the post and choose the one label that fits it best. "
    "A post about rumoured redundancies counts as layoff."
)
ev2 = clf2.evaluate(dev, target=0.9)
ev2

3/3  0 live, 3 cached, 0 errors, cost $0.0000


Evaluation: 3 rows, 0 pending rows skipped, 0 disagreements
accuracy 1.0000 PASS (target 0.9000)
macro_f1 1.0000, micro_f1 1.0000
label      precision  recall  f1    gold  predicted
layoff          1.00    1.00  1.00     1          1
burnout         1.00    1.00  1.00     2          2
off_topic       0.00    0.00  0.00     0          0
sources: goldlabel-ui 3, imported 0, unstamped 0

## 6. The test split, once

Draw and label a separate test sample, and evaluate it once, at the end, to report the number.

In [10]:
test = gold.draw_sample("test", n=2)

In [11]:
gl.label(test, taxonomy)

All 2 rows in test are labelled; nothing to label. Pass force_open_ui=True to review or relabel.


id,text,labels,sample,labelled by
layoff-2,"My whole team was cut in the reorg, and I am job hunting now.",layoff,test,goldlabel-ui
layoff-3,Rumours of redundancies are going around the office.,layoff,test,goldlabel-ui


In [12]:
final = clf2.evaluate(test, target=0.9)
final

2/2  0 live, 2 cached, 0 errors, cost $0.0000


Evaluation: 2 rows, 0 pending rows skipped, 0 disagreements
accuracy 1.0000 PASS (target 0.9000)
macro_f1 1.0000, micro_f1 1.0000
label      precision  recall  f1    gold  predicted
layoff          1.00    1.00  1.00     2          2
burnout         0.00    0.00  0.00     0          0
off_topic       0.00    0.00  0.00     0          0
sources: goldlabel-ui 2, imported 0, unstamped 0

In [13]:
test  # its summary says how often this held-out sample was evaluated

id,text,labels,sample,labelled by
layoff-2,"My whole team was cut in the reorg, and I am job hunting now.",layoff,test,goldlabel-ui
layoff-3,Rumours of redundancies are going around the office.,layoff,test,goldlabel-ui


## 7. Pick a cutoff, price the run, run it

`band_sample` classifies a few items per similarity band and shows how much of each is off topic.
You choose the cutoff. `estimate` calls no model; `unknown` cost shows as `?`, never as `$0`.

In [14]:
report = clf2.band_sample(candidates, per_band=5)
report

band,n,off_topic,rate
0.650-0.675,1,0,0%
0.675-0.700,1,0,0%
0.750+,3,0,0%


In [15]:
chosen = candidates.above(0.70)
clf2.estimate(chosen)

Estimate: 3 items, 3 cached, 0 live calls
cost ~$0.0000, time ~0s

In [16]:
# max_cost refuses before spending anything if the estimate is over it or the price is unknown.
results = clf2.classify(chosen, out="results.jsonl", max_cost=1.00)
results.to_polars()

id,text,labels,reasoning,run,run_model,run_at,max_similarity,best_seed
str,str,list[str],str,str,str,str,f64,str
"""layoff-1""","""I got laid off on Monday after…","[""layoff""]","""The author directly says they …","""c6e5f57b496c967bf383a803f2b39c…","""openai:gpt-6-luna""","""2026-11-10T21:05:23.596206+00:…",0.95,"""I just lost my job"""
"""layoff-2""","""My whole team was cut in the r…","[""layoff""]","""The author says their whole te…","""c6e5f57b496c967bf383a803f2b39c…","""openai:gpt-6-luna""","""2026-11-10T21:05:23.597132+00:…",0.9,"""I just lost my job"""
"""layoff-3""","""Rumours of redundancies are go…","[""layoff""]","""The post says rumours of redun…","""c6e5f57b496c967bf383a803f2b39c…","""openai:gpt-6-luna""","""2026-11-10T21:05:23.597839+00:…",0.8,"""I just lost my job"""


## 8. Write the methodology

`get_methodology` describes what was done in plain English, from the objects you give it and
the files beside them (the gold file and its evaluation log). It calls no model and costs
nothing, and it says "not recorded" for anything a file does not hold. Pass `chosen`, the list
you classified, so the cutoff is stated. `str(m)` is Markdown to paste into an article, and
`m.warnings` lists real problems, such as a score older than a label revision. The appendix
lists every gold row, text included; pass `include_gold_table=False` to leave that out.

In [17]:
m = gl.get_methodology(candidates=chosen, gold=gold, classifier=clf2, results=results)
print(m)

## Methodology

### Corpus and candidates

We searched the table file corpus.jsonl (text column text, id column id) for items similar to 1 example phrase ("I just lost my job"). We embedded each phrase with hand-built-2d and kept every item whose highest similarity to any phrase was at least 0.6. We then kept the items with similarity of at least 0.7, which left 3.

### Labels

Each item received exactly one of these labels: `layoff`, `burnout`, or the built-in `off_topic`, which means the item fits none of them.

### Gold labels

People labelled items by hand, and we measured the classifier against their labels. The gold dataset "default" holds 2 samples:

- dev: 3 rows, 3 labelled, 0 pending.
- test: 2 rows, 2 labelled, 0 pending.

### Classifier

The classifier uses the model `openai:gpt-6-luna`. We evaluated 2 prompts against the gold labels.

### Evaluation

We scored the classifier against the gold labels:

- On dev, accuracy was 1.00 across 3 labelled rows, meeting the 0.9 targe

In [18]:
m.warnings

()

## 9. Keep the classifier

A saved bundle (model, taxonomy, prompt, the final evaluation) is plain files to commit. A
pipeline loads it with `gl.Classifier.load("classifiers/layoffs")`, no gold and no UI needed.

In [19]:
clf2.save("classifiers/layoffs", evaluation=final)